# Cortex Search: Access Control and RBAC Runbook

**Purpose**: Examples and templates for implementing RBAC for Cortex Search services.

**Target Audience**: Snowflake Administrators

**Last Updated**: March 2026

**Version**: 2.0

---

## Overview

This runbook covers creating Cortex Search services with role separation (creator, user, operator), querying with `SEARCH_PREVIEW`, and managing service operations (suspend/resume/refresh).

---
## 1. Configuration Variables

Set these variables at the beginning to customize the runbook for your environment.


In [ ]:
USE SECONDARY ROLES NONE;

-- Configuration Variables
-- Modify these values according to your environment

SET DATABASE_NAME = 'RUNBOOKS_DB';
SET SCHEMA_NAME = 'CORTEX_SEARCH';
SET WAREHOUSE_NAME = 'COMPUTE_WH';

-- Role names for RBAC
SET SERVICE_CREATOR_ROLE = 'CORTEX_SERVICE_CREATOR_ROLE';
SET SERVICE_USER_ROLE = 'CORTEX_SERVICE_USER_ROLE';
SET SERVICE_OPERATOR_ROLE = 'CORTEX_SERVICE_OPERATOR_ROLE';

-- Sample data table and service names
SET SOURCE_TABLE = 'PRODUCT_DOCS';
SET SEARCH_SERVICE_NAME = 'PRODUCT_SEARCH_SERVICE';

-- Derived fully qualified names (for use with IDENTIFIER function)
SET FULL_SCHEMA_NAME = $DATABASE_NAME || '.' || $SCHEMA_NAME;
SET FULL_TABLE_NAME = $DATABASE_NAME || '.' || $SCHEMA_NAME || '.' || $SOURCE_TABLE;
SET FULL_SERVICE_NAME = $DATABASE_NAME || '.' || $SCHEMA_NAME || '.' || $SEARCH_SERVICE_NAME;

-- Display configuration
SELECT 'Configuration Set Successfully' AS STATUS,
       $DATABASE_NAME AS DATABASE,
       $SCHEMA_NAME AS SCHEMA,
       $WAREHOUSE_NAME AS WAREHOUSE,
       $FULL_SCHEMA_NAME AS FULL_SCHEMA,
       $FULL_SERVICE_NAME AS FULL_SERVICE_NAME;


---
## 2. Account Level Access Controls

### 2.1 Understanding Cortex Database Roles

Snowflake provides two database roles in the `SNOWFLAKE` database for Cortex functionality:

- **`SNOWFLAKE.CORTEX_USER`**: Grants access to all Cortex features including LLM functions, embedding functions, and Cortex Search
- **`SNOWFLAKE.CORTEX_EMBED_USER`**: Grants access only to embedding functions (AI_EMBED, EMBED_TEXT_768, EMBED_TEXT_1024) and Cortex Search creation

**Default Behavior:** By default, `CORTEX_USER` is granted to the `PUBLIC` role, giving all users access to Cortex features.

### 2.2 Revoking Default Access (Recommended for Production)

For production environments, it's recommended to revoke default access and grant privileges selectively.


In [ ]:
-- Switch to ACCOUNTADMIN role (required for database role management)
USE ROLE ACCOUNTADMIN;

-- Revoke CORTEX_USER from PUBLIC role to restrict default access
REVOKE DATABASE ROLE SNOWFLAKE.CORTEX_USER FROM ROLE PUBLIC;

### 2.3 Setting Up Infrastructure

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Create database
CREATE DATABASE IF NOT EXISTS IDENTIFIER($DATABASE_NAME);

-- Create schema (using fully qualified name)
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($FULL_SCHEMA_NAME);

-- Create warehouse for Cortex Search operations
CREATE WAREHOUSE IF NOT EXISTS IDENTIFIER($WAREHOUSE_NAME)
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
    COMMENT = 'Warehouse for Cortex Search service refresh operations';

---
## 3. RBAC and Embedding Model Controls

### 3.1 Creating Custom Roles

Role hierarchy for Cortex Search:
1. **Service Creator Role**: Create and manage search services
2. **Service User Role**: Query search services
3. **Service Operator Role**: Suspend/resume services

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Create custom roles
CREATE ROLE IF NOT EXISTS IDENTIFIER($SERVICE_CREATOR_ROLE)
    COMMENT = 'Role for creating Cortex Search services';

CREATE ROLE IF NOT EXISTS IDENTIFIER($SERVICE_USER_ROLE)
    COMMENT = 'Role for querying Cortex Search services';

CREATE ROLE IF NOT EXISTS IDENTIFIER($SERVICE_OPERATOR_ROLE)
    COMMENT = 'Role for operating (suspend/resume) Cortex Search services';

-- Set up role hierarchy
GRANT ROLE IDENTIFIER($SERVICE_USER_ROLE) TO ROLE IDENTIFIER($SERVICE_OPERATOR_ROLE);
GRANT ROLE IDENTIFIER($SERVICE_OPERATOR_ROLE) TO ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);
GRANT ROLE IDENTIFIER($SERVICE_CREATOR_ROLE) TO ROLE SYSADMIN;

### 3.2 Granting Cortex Database Roles

To create Cortex Search services, roles need access to embedding functions via `SNOWFLAKE.CORTEX_USER` or `SNOWFLAKE.CORTEX_EMBED_USER`. We grant `CORTEX_EMBED_USER` to the creator role for least privilege.

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Grant embedding access to service creator role
-- This is REQUIRED to create Cortex Search services
GRANT DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER TO ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);

-- Optional: If you want service users to also use LLM functions, grant CORTEX_USER
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE IDENTIFIER($SERVICE_USER_ROLE);

### 3.3 Granting Infrastructure Privileges

In [ ]:
USE ROLE ACCOUNTADMIN;

-- Grant database and schema privileges to service creator role
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_NAME) TO ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);
GRANT CREATE CORTEX SEARCH SERVICE ON SCHEMA IDENTIFIER($FULL_SCHEMA_NAME) TO ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);

-- Grant database and schema privileges to service user role
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($SERVICE_USER_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_NAME) TO ROLE IDENTIFIER($SERVICE_USER_ROLE);

-- Grant database and schema privileges to operator role (will be granted opperator permissions later)
GRANT USAGE ON DATABASE IDENTIFIER($DATABASE_NAME) TO ROLE IDENTIFIER($SERVICE_OPERATOR_ROLE);
GRANT USAGE ON SCHEMA IDENTIFIER($FULL_SCHEMA_NAME) TO ROLE IDENTIFIER($SERVICE_OPERATOR_ROLE);

-- Grant warehouse usage to all roles that need it
GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);

### 3.4 Privilege Summary

In [ ]:
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);
-- View privileges for each custom role
--SHOW GRANTS TO ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);
SHOW GRANTS TO ROLE IDENTIFIER($SERVICE_USER_ROLE);
--SHOW GRANTS TO ROLE IDENTIFIER($SERVICE_OPERATOR_ROLE);

---
## 4. Creating Cortex Search Services

### 4.1 Create Sample Data

In [ ]:
USE ROLE ACCOUNTADMIN;
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

-- Create sample table with product documentation
CREATE OR REPLACE TABLE IDENTIFIER($SOURCE_TABLE) (
    doc_id NUMBER AUTOINCREMENT,
    product_name VARCHAR,
    category VARCHAR,
    documentation TEXT,
    last_updated TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

-- Enable change tracking (REQUIRED for Cortex Search)
ALTER TABLE IDENTIFIER($SOURCE_TABLE) SET CHANGE_TRACKING = TRUE;

-- Insert sample data
INSERT INTO IDENTIFIER($SOURCE_TABLE) (product_name, category, documentation)
VALUES
    ('Snowflake Data Cloud', 'Platform', 'Snowflake is a cloud-based data platform that enables data warehousing, data lakes, data engineering, data science, and data application development.'),
    ('Cortex Search', 'AI/ML', 'Cortex Search is a fully managed service that provides hybrid search capabilities combining vector search, keyword search, and semantic reranking.'),
    ('Snowpark', 'Development', 'Snowpark is a developer framework that enables data engineers and data scientists to build and deploy data pipelines, ML models, and data applications in Snowflake.'),
    ('Streamlit', 'Visualization', 'Streamlit in Snowflake allows you to build and share data applications directly within Snowflake without managing infrastructure.'),
    ('Dynamic Tables', 'Data Engineering', 'Dynamic Tables are a declarative way to define data pipelines in Snowflake that automatically refresh based on changes to source data.');

    -- Grant SELECT privilege to service creator role
GRANT SELECT ON TABLE IDENTIFIER($SOURCE_TABLE) TO ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);

SELECT * FROM IDENTIFIER($SOURCE_TABLE);

### 4.2 Create Cortex Search Service

**Required Privileges**: `CREATE CORTEX SEARCH SERVICE` on schema, `SELECT` on source table, `USAGE` on warehouse, and `CORTEX_USER` or `CORTEX_EMBED_USER` database role.

In [ ]:
USE ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);

-- Create Cortex Search Service
CREATE OR REPLACE CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME)
    ON documentation                          -- Column to search on
    ATTRIBUTES product_name, category         -- Columns to filter on
    WAREHOUSE = 'COMPUTE_WH'                  -- Note: Doesn't work with session variable
    TARGET_LAG = '1 hour'                     -- Maximum lag for data freshness
    EMBEDDING_MODEL = 'snowflake-arctic-embed-m-v1.5'  -- Default embedding model
    COMMENT = 'Search service for product documentation'
AS (
    SELECT
        doc_id,
        product_name,
        category,
        documentation,
        last_updated
    FROM PRODUCT_DOCS
);

### 4.3 View Service Details


In [ ]:
-- Describe the Cortex Search Service
DESCRIBE CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME);

-- Show all Cortex Search Services in the schema
SHOW CORTEX SEARCH SERVICES IN SCHEMA IDENTIFIER($FULL_SCHEMA_NAME);


---
## 5. Querying with Proper Privileges

### 5.1 Understanding Query Privileges

**Required**: `USAGE` on the Cortex Search Service, database, and schema.

Cortex Search Services run with **owner's rights** -- users with USAGE on the service can query all indexed data regardless of table privileges.

### 5.2 Grant Query Privilege

In [ ]:
USE ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);

-- Grant USAGE privilege on the service to the user role
GRANT USAGE ON CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME) 
    TO ROLE IDENTIFIER($SERVICE_USER_ROLE);

USE ROLE ACCOUNTADMIN;

GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE IDENTIFIER($SERVICE_USER_ROLE); -- to call cortex search service

GRANT USAGE ON WAREHOUSE IDENTIFIER($WAREHOUSE_NAME) TO ROLE IDENTIFIER($SERVICE_USER_ROLE); -- Warehouse in session needed to call function.

### 5.3 Query the Service as a User

In [ ]:
USE ROLE IDENTIFIER($SERVICE_USER_ROLE);
USE DATABASE IDENTIFIER($DATABASE_NAME);
USE SCHEMA IDENTIFIER($SCHEMA_NAME);
USE WAREHOUSE IDENTIFIER($WAREHOUSE_NAME);

-- Query the Cortex Search Service

SELECT 
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'PRODUCT_SEARCH_SERVICE', --IDENTIFIER($SEARCH_SERVICE_NAME)
       '{
            "query": "machine learning and data science",
            "columns": ["product_name", "category", "documentation"],
            "limit": 5
        }'
    ) AS search_results;

### 5.4 Query with Filters

In [ ]:
-- Query with category filter
SELECT 
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'PRODUCT_SEARCH_SERVICE',
        '{
            "query": "data platform",
            "columns": ["product_name", "category", "documentation"],
            "filter": {"@eq": {"category": "Platform"}},
            "limit": 5
        }'
    ) AS search_results;

---
## 6. Managing Service Operations

### 6.1 OPERATE Privilege

Cortex Search services have two processes:
- **INDEXING**: Refreshes the search index from source data
- **SERVING**: Handles query requests

### 6.2 Grant OPERATE Privilege

In [ ]:
USE ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);

-- Grant OPERATE privilege to the operator role
GRANT OPERATE ON CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME)
    TO ROLE IDENTIFIER($SERVICE_OPERATOR_ROLE);

### 6.3 Suspend and Resume Operations

In [ ]:
USE ROLE IDENTIFIER($SERVICE_OPERATOR_ROLE);

-- Suspend indexing (stops data refresh, saves costs)
ALTER CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME) SUSPEND INDEXING;

-- Check service status
DESCRIBE CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME);


In [ ]:
-- Resume indexing
ALTER CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME) RESUME INDEXING;

In [ ]:
-- Suspend both indexing and serving
ALTER CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME) SUSPEND;

In [ ]:
-- Resume both indexing and serving
ALTER CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME) RESUME;

### 6.4 Manual Refresh

In [ ]:
USE ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);

-- Trigger manual refresh of the search index
ALTER CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME) REFRESH;

### 6.5 Modify Service Properties

In [ ]:
USE ROLE IDENTIFIER($SERVICE_CREATOR_ROLE);

-- Modify TARGET_LAG and WAREHOUSE
ALTER CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME)
    SET TARGET_LAG = '30 minutes'
        COMMENT = 'Updated: Reduced target lag for faster data refresh';

In [ ]:
-- View grants on Cortex Search services
SHOW GRANTS ON CORTEX SEARCH SERVICE IDENTIFIER($SEARCH_SERVICE_NAME);

In [ ]:
-- View who has Cortex database roles
SHOW GRANTS OF DATABASE ROLE SNOWFLAKE.CORTEX_USER;
SHOW GRANTS OF DATABASE ROLE SNOWFLAKE.CORTEX_EMBED_USER;

---

## Additional Resources

- [Cortex Search Overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)
- [Querying Cortex Search Services](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/query-cortex-search-service)
- [Row Access Policies](https://docs.snowflake.com/en/user-guide/security-row-intro)

---

**Version:** 2.0  
**Last Updated:** March 2026

---

## 7. Cleanup (Optional)

Run these commands to clean up all resources created in this runbook.

In [ ]:
-- Cleanup: Drop all created objects
--USE ROLE ACCOUNTADMIN;

-- Drop Cortex Search Service
--DROP CORTEX SEARCH SERVICE IF EXISTS IDENTIFIER($SEARCH_SERVICE_NAME);

-- Drop sample data table
--DROP TABLE IF EXISTS IDENTIFIER($SOURCE_TABLE);

-- Drop warehouse
--DROP WAREHOUSE IF EXISTS IDENTIFIER($WAREHOUSE_NAME);

-- Drop custom roles
--DROP ROLE IF EXISTS IDENTIFIER($SERVICE_CREATOR_ROLE);
--DROP ROLE IF EXISTS IDENTIFIER($SERVICE_USER_ROLE);
--DROP ROLE IF EXISTS IDENTIFIER($SERVICE_OPERATOR_ROLE);

-- Drop schema and database (commented out for safety)
-- DROP SCHEMA IF EXISTS IDENTIFIER($FULL_SCHEMA_NAME);
-- DROP DATABASE IF EXISTS IDENTIFIER($DATABASE_NAME);

-- Re-grant CORTEX_USER to PUBLIC if needed
-- GRANT DATABASE ROLE SNOWFLAKE.CORTEX_USER TO ROLE PUBLIC;